In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

print("train score:", model.score(X_train, y_train))   # usually 1.0, too good
print("test score:", model.score(X_test, y_test))      # the honest one

train score: 1.0
test score: 0.9122807017543859


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(scores)
print("mean:", scores.mean())

[0.97368421 0.94736842 0.96491228 0.99122807 0.99115044]
mean: 0.9736686849868033


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "tree": DecisionTreeClassifier(random_state=42),
}
for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=cv, scoring="f1")
    print(name, round(scores.mean(), 3))

logistic 0.979
tree 0.93


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for depth in [2, None]:
    res = cross_validate(DecisionTreeClassifier(max_depth=depth, random_state=42),
                         X, y, cv=cv, return_train_score=True)
    print("max_depth:", depth,
          "| train:", round(res["train_score"].mean(), 3),
          "| validation:", round(res["test_score"].mean(), 3))

max_depth: 2 | train: 0.956 | validation: 0.905
max_depth: None | train: 1.0 | validation: 0.91


In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print("mean:", round(scores.mean(), 3), "| std:", round(scores.std(), 3))

mean: 0.976 | std: 0.013


In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(C=1.0, max_iter=5000)   # C is a hyperparameter (you set it)
model.fit(X_train, y_train)

print(model.get_params()["C"])    # hyperparameter
print(model.coef_.shape)          # learned parameters (weights)

1.0
(1, 30)


In [5]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {"max_depth": [2, 4, 6], "min_samples_leaf": [1, 5, 10]}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)
print("test:", grid.score(X_test, y_test))   # final honest check

{'max_depth': 4, 'min_samples_leaf': 1}
0.9384615384615385
test: 0.9385964912280702


In [6]:
from scipy.stats import randint
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_dist = {"max_depth": randint(2, 20), "min_samples_leaf": randint(1, 30)}
search = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), param_dist,
                            n_iter=15, cv=5, scoring="accuracy", random_state=42)
search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)
print("test:", search.score(X_test, y_test))

{'max_depth': 5, 'min_samples_leaf': 8}
0.9208791208791208
test: 0.9473684210526315


In [7]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier())])
grid = GridSearchCV(pipe, {"model__n_neighbors": [3, 5, 7, 9, 11]}, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print(grid.best_params_)
print("final test score:", grid.score(X_test, y_test))   # looked at only once

{'model__n_neighbors': 7}
final test score: 0.9795918367346939


In [8]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

base = DecisionTreeClassifier(random_state=42)
base_cv = cross_val_score(base, X_train, y_train, cv=5, scoring="f1").mean()

grid = GridSearchCV(base, {"max_depth": [2, 3, 4, 6], "min_samples_leaf": [1, 5, 10]}, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print("baseline CV F1:", round(base_cv, 3))
print("tuned CV F1:", round(grid.best_score_, 3))
print("tuned test F1:", round(grid.score(X_test, y_test), 3))

baseline CV F1: 0.928
tuned CV F1: 0.951
tuned test F1: 0.951


In [9]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

base = DecisionTreeClassifier(random_state=42)
base_cv = cross_val_score(base, X_train, y_train, cv=5, scoring="f1").mean()

grid = GridSearchCV(base, {"max_depth": [2, 3, 4, 6], "min_samples_leaf": [1, 5, 10]}, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print("baseline CV F1:", round(base_cv, 3))
print("tuned CV F1:", round(grid.best_score_, 3))
print("tuned test F1:", round(grid.score(X_test, y_test), 3))

baseline CV F1: 0.928
tuned CV F1: 0.951
tuned test F1: 0.951


In [10]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
])
pipe.fit(X_train, y_train)
print(pipe.predict(X_test)[:10])

[0 1 0 0 0 1 1 0 0 0]


In [11]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
])

scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1")
print("CV F1:", round(scores.mean(), 3), "±", round(scores.std(), 3))

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

CV F1: 0.967 ± 0.013
[[39  3]
 [ 2 70]]
              precision    recall  f1-score   support

           0       0.95      0.93      0.94        42
           1       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [12]:
import joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("select", SelectKBest(f_classif)),
    ("model", RandomForestClassifier(random_state=42)),
])
param_grid = {"select__k": [5, 10, 20], "model__max_depth": [3, 5, None]}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1")
grid.fit(X_train, y_train)

print(grid.best_params_)
print("CV F1:", round(grid.best_score_, 3))
print("test F1:", round(grid.score(X_test, y_test), 3))

joblib.dump(grid.best_estimator_, "best_pipeline.joblib")   # save the trained pipeline

{'model__max_depth': None, 'select__k': 20}
CV F1: 0.965
test F1: 0.966


['best_pipeline.joblib']